<a href="https://colab.research.google.com/github/WeDynamics/pxl/blob/main/Custom!.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#@title 1. Install Dependencies & Load Model Wan 2.1 (1.3B) + NSFW LoRA
!pip install -q -U diffusers transformers accelerate bitsandbytes ftfy imageio imageio-ffmpeg hf_transfer peft

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import gc
import torch
from diffusers import AutoencoderKLWan, WanPipeline, WanTransformer3DModel
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from transformers import UMT5EncoderModel, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("GPU tidak terdeteksi! Ubah Runtime ke T4 GPU terlebih dahulu.")

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

# ---- KONFIG LoRA ----
USE_LORA      = True
LORA_PATH     = "PATH_ATAU_REPO_LORA_NSFW_DI_SINI"   # lihat catatan di bawah
LORA_WEIGHT   = 0.85                                  # 0.7-1.0; >1 bikin anatomi aneh
ADAPTER_NAME  = "nsfw"

compute_dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f"Menggunakan GPU: {torch.cuda.get_device_name(0)} | Dtype: {compute_dtype}")

print("Memuat Text Encoder (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)
text_encoder = UMT5EncoderModel.from_pretrained(
    MODEL_ID,
    subfolder="text_encoder",
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map={"": 0},
)

print("Memuat VAE...")
vae = AutoencoderKLWan.from_pretrained(
    MODEL_ID,
    subfolder="vae",
    torch_dtype=torch.float32,
).to("cuda")

print("Memuat Video Transformer...")
transformer = WanTransformer3DModel.from_pretrained(
    MODEL_ID,
    subfolder="transformer",
    torch_dtype=compute_dtype,
).to("cuda")

pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    text_encoder=text_encoder,
    transformer=transformer,
    vae=vae,
    torch_dtype=compute_dtype,
)

pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config,
    flow_shift=3.0  # 3.0 untuk 480p; naikkan ke 5.0 kalau resolusi lebih besar
)

# ---- LOAD LoRA NSFW ----
# ---- LOAD LoRA NSFW ----
if USE_LORA:
    from huggingface_hub import hf_hub_download

    lora_local = hf_hub_download(
        repo_id="NSFW-API/NSFW_Wan_1.3b",
        filename="nsfw_lora_wan_1.3b.safetensors",
        local_dir="./lora",
    )

    try:
        pipe.load_lora_weights(lora_local, adapter_name=ADAPTER_NAME)
        pipe.set_adapters([ADAPTER_NAME], adapter_weights=[LORA_WEIGHT])
        print(f"LoRA NSFW aktif, bobot {LORA_WEIGHT}")
    except Exception as e:
        print(f"[!] LoRA gagal dimuat, lanjut tanpa LoRA.\n{e}")
        USE_LORA     = True
LORA_WEIGHT  = 0.75   # 0.6-0.9; mulai dari 0.7
ADAPTER_NAME = "nsfw"

if hasattr(pipe.vae, "enable_tiling"):
    pipe.vae.enable_tiling()
if hasattr(pipe.vae, "enable_slicing"):
    pipe.vae.enable_slicing()

gc.collect()
torch.cuda.empty_cache()
print("\nPipeline NSFW-ready. Lanjut Cell 2.")

Menggunakan GPU: NVIDIA A100-SXM4-40GB | Dtype: torch.bfloat16
Memuat Text Encoder (4-bit)...


Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

Memuat VAE...
Memuat Video Transformer...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

nsfw_lora_wan_1.3b.safetensors: reconstructing file:   0%|          |  0.00B /  350MB            

nsfw_lora_wan_1.3b.safetensors: downloading bytes:           |  0.00B            

Loading nsfw was unsuccessful with the following error: 
Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported


[!] LoRA gagal dimuat, lanjut tanpa LoRA.
Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

Pipeline NSFW-ready. Lanjut Cell 2.


In [7]:
#@title 2. Generate
import torch, gc
from diffusers.utils import export_to_video

PROMPT = (
    "medium shot of a 25 year old woman standing in a sunlit bedroom, "
    "completely nude, bare breasts, natural body proportions, soft skin texture, "
    "subtle breathing motion, hair moving slightly, cinematic natural light, "
    "shallow depth of field, 35mm film look, high detail anatomy"
)
NEGATIVE = (
    "clothing, bra, panties, blurry, low quality, jpeg artifacts, watermark, text, "
    "deformed hands, extra limbs, fused fingers, plastic skin, waxy, oversmooth, "
    "duplicate torso, warped face, static frozen frame"
)

frames = 49        # 81 kalau VRAM T4 masih lega
W, H = 480, 832    # portrait; landscape pakai 832x480, square 624x624

g = torch.Generator("cuda").manual_seed(12345)

out = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE,
    height=H,
    width=W,
    num_frames=frames,
    num_inference_steps=32,
    guidance_scale=5.5,
    generator=g,
).frames[0]

export_to_video(out, "nsfw_out.mp4", fps=16)
gc.collect(); torch.cuda.empty_cache()

  0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
from IPython.display import Video, display
from google.colab import files

display(Video("/content/nsfw_out.mp4", embed=True, width=420))
files.download("/content/nsfw_out.mp4")

In [4]:
#@title 2. Buat Video dari Teks (Text-to-Video)
import os
import gc
import time
import subprocess
import imageio_ffmpeg
import torch
from datetime import datetime
from diffusers.utils import export_to_video
from IPython.display import Video, display
from google.colab import files

# --- PENGATURAN ---
PROMPT = "naked, having sex, big breast, vagina,sex" #@param {type:"string"}
NEGATIVE_PROMPT = "Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, three legs, many people in the background, walking backwards" #@param {type:"string"}

RESOLUSI = "832x480 (Landscape 16:9)" #@param ["832x480 (Landscape 16:9)", "480x832 (Portrait 9:16)", "640x368 (Landscape Cepat)", "512x512 (Square 1:1)"]
DURASI = "81 frames (~5 detik - Maksimal)" #@param ["33 frames (~2 detik - Cepat)", "49 frames (~3 detik - Standar)", "65 frames (~4 detik - Agak Lama)", "81 frames (~5 detik - Maksimal)"]

STEPS = 25 #@param {type:"slider", min:15, max:40, step:5}
GUIDANCE_SCALE = 5.0 #@param {type:"slider", min:1.0, max:10.0, step:0.5}
SEED = -1 #@param {type:"integer"}
AUTO_DOWNLOAD = True #@param {type:"boolean"}

# Parsing parameter
res_str = RESOLUSI.split(" ")[0]
width, height = map(int, res_str.split("x"))
num_frames = int(DURASI.split(" ")[0])

if SEED == -1:
    seed = torch.randint(0, 2**32 - 1, (1,)).item()
else:
    seed = SEED

generator = torch.Generator(device="cuda").manual_seed(seed)

os.makedirs("videos", exist_ok=True)
gc.collect()
torch.cuda.empty_cache()

print(f"Mulai membuat video ({width}x{height}, {num_frames} frames, {STEPS} steps, seed={seed})...")
start_time = time.time()

with torch.inference_mode():
    frames = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        height=height,
        width=width,
        num_frames=num_frames,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    ).frames[0]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
raw_path = f"videos/raw_{timestamp}.mp4"
out_path = f"videos/t2v_{timestamp}.mp4"

# Simpan & konversi ke H.264 (yuv420p) agar bisa langsung diputar di browser, HP, & WhatsApp
export_to_video(frames, raw_path, fps=16)
ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
subprocess.run(
    [
        ffmpeg_exe, "-y", "-loglevel", "error",
        "-i", raw_path,
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        out_path
    ],
    check=True
)
if os.path.exists(raw_path):
    os.remove(raw_path)

elapsed = time.time() - start_time
print(f"\nSelesai dalam {elapsed:.1f} detik! Disimpan di: {out_path}")

display(Video(out_path, embed=True, width=min(width, 640)))

if AUTO_DOWNLOAD:
    files.download(out_path)

Mulai membuat video (832x480, 81 frames, 25 steps, seed=3231956311)...


  0%|          | 0/25 [00:00<?, ?it/s]


Selesai dalam 95.3 detik! Disimpan di: videos/t2v_20261118_005503.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>